# ZeroGPT Kaggle runner

This notebook reproduces the repository's corpus preparation, character bigram baseline, and tiny GPT run. Make the repository available as a Kaggle dataset or upload it into the working directory before running the cells.

The NumPy implementation runs on the Kaggle CPU; the T4 does not accelerate these operations.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

def _is_repo(path: Path) -> bool:
    return (path / 'roadmap.md').exists() and (path / 'dataset.txt').exists()

def _extract_zip(zip_path: Path) -> Path:
    target = zip_path.parent / f"{zip_path.stem}_extracted"
    if target.exists():
        shutil.rmtree(target)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)
    subdirs = [p for p in target.iterdir() if p.is_dir()]
    if len(subdirs) == 1 and not _is_repo(target):
        return subdirs[0]
    return target

def _scan(base: Path) -> list[Path]:
    """Recursively collect repo directories, extracting any zips found."""
    if not base.exists():
        return []
    repos: list[Path] = []
    for entry in base.iterdir():
        if entry.is_dir():
            repos.extend(_scan(entry))
        elif entry.is_file() and entry.suffix.lower() == '.zip':
            extracted = _extract_zip(entry)
            repos.extend(_scan(extracted))
    if _is_repo(base):
        repos.append(base)
    return repos

candidates = [Path.cwd(), Path('/kaggle/working/ZeroGPT'), Path('/kaggle/input/zerogpt')]
candidates += _scan(Path('/kaggle/working')) + _scan(Path('/kaggle/input'))
source_root = next((p for p in candidates if (p / 'roadmap.md').exists() and (p / 'dataset.txt').exists()), None)
if source_root is None:
    raise FileNotFoundError('Make the ZeroGPT repository available in the Kaggle working directory or as a dataset.')
if str(source_root).startswith('/kaggle/input/'):
    ROOT = Path('/kaggle/working/ZeroGPT')
    if ROOT.exists():
        shutil.rmtree(ROOT)
    shutil.copytree(source_root, ROOT)
else:
    ROOT = source_root
os.chdir(ROOT)
print('Repository root:', ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt

## Prepare the document-ordered split

In [ ]:
!python data/prepare.py

## Train the count-based baseline

In [ ]:
!python scripts/train_bigram.py --output-dir runs/bigram --tokenizer-type byte_pair

## Train the tiny GPT

The configuration matches the saved showcase run: seed 1337, 6,000 steps, 2 decoder layers, 8 heads, width 64, context length 64, with a 200-step warmup and cosine decay learning-rate schedule.

In [ ]:
!python scripts/train_gpt.py --steps 6000 --batch-size 16 --eval-batch-size 64 --eval-interval 100 --warmup-steps 200 --sample-tokens 400 --tokenizer-type byte_pair --output-dir runs/gpt

In [ ]:
import json
from pathlib import Path

bigram = json.loads(Path('runs/bigram/results.json').read_text())
gpt = json.loads(Path('runs/gpt/results.json').read_text())
print('Bigram validation loss:', bigram['validation_loss_nats'])
print('GPT validation loss:', gpt['validation_loss_nats'])
print('GPT beats bigram:', gpt['beats_bigram'])
print('GPT checkpoint max logit difference:', gpt['checkpoint_max_logit_difference'])
print(Path('runs/gpt/sample.txt').read_text())

In [ ]:
import shutil

artifact = shutil.make_archive('zerogpt_kaggle_runs', 'zip', root_dir='runs')
print('Saved Kaggle artifact:', artifact)